# 08 — Canonical resumable training campaign

> **Status:** empty implementation skeleton.

- **Mapped issue:** [#16](https://github.com/majorgilles/transformer-2017-reproduction/issues/16)
- **Depends on:** `07_gpu_calibration_freeze.ipynb` / issue #15.


In [1]:
#| default_exp operations


## Goal

Run the frozen campaign across sessions and select using validation only.


## Build token-budget batches from WMT text

Training examples are encoded in bounded buffers rather than loading four
million tokenized pairs into memory. Each stage below has one responsibility:
padding a batch, grouping one buffer, or streaming examples.

The iterator ultimately yields two integer tensors using the same dimension names as the earlier notebooks:

```text
source_token_ids: (batch, source_length)
target_token_ids: (batch, target_length + 1)
```

The complete target includes `BOS` and `EOS`. The training loop later shifts it into decoder inputs and next-token labels, both shaped `(batch, target_length)`.


In [2]:
#| export
from collections.abc import Iterator, Sequence

import torch
from tokenizers.tokenizers import Tokenizer
from torch.nn.utils.rnn import pad_sequence

from transformer_2017_reproduction.calibration import CANONICAL_CAMPAIGN
from transformer_2017_reproduction.data import (
    ParallelExample,
    iter_manifest_examples,
    load_manifest,
)
from transformer_2017_reproduction.environment import PROJECT_ROOT
from transformer_2017_reproduction.optimization import make_token_budget_batches

### Pad one selected batch

The token-budget batcher chooses which examples belong together. Collation then pads their source and target sequences independently so each becomes a rectangular tensor. Rows are separate sentence pairs; columns are token positions.

For example, two selected source sequences of lengths four and three become:

```text
before padding                 after padding: shape (batch=2, source_length=4)
[BOS, 11, 12, EOS]             [[BOS, 11, 12, EOS],
[BOS, 21, EOS]                  [BOS, 21, EOS, PAD]]
```

Targets are padded separately because their longest sequence may have a different length. `_collate_token_batch` therefore returns `(source_token_ids, target_token_ids)`, not one combined tensor.


In [3]:
#| export
def _collate_token_batch(
    batch: Sequence[tuple[Sequence[int], Sequence[int]]],
    pad_token_id: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Pad encoded source-target pairs into rectangular tensors."""
    # Before padding, each source is a variable-length sequence: (source_length,).
    # After padding, source_token_ids has shape (batch, source_length).
    source_token_ids = pad_sequence(
        [torch.tensor(source_ids, dtype=torch.long) for source_ids, _ in batch],
        batch_first=True,
        padding_value=pad_token_id,
    )

    # Before padding, each complete target has shape (target_length,).
    # After padding, target_token_ids has shape (batch, target_length + 1).
    target_token_ids = pad_sequence(
        [torch.tensor(target_ids, dtype=torch.long) for _, target_ids in batch],
        batch_first=True,
        padding_value=pad_token_id,
    )

    return source_token_ids, target_token_ids

### Group one encoded buffer

Examples of similar lengths are placed near each other before batching. This reduces padding while keeping memory bounded to one buffer.

A simplified encoded buffer might contain lengths `(4, 4)`, `(20, 18)`, `(5, 6)`, and `(19, 21)`, where each pair is `(source length, target length)`. Sorting changes their processing order to approximately `(4, 4)`, `(5, 6)`, `(20, 18)`, `(19, 21)`. Short rows then share batches with short rows instead of receiving enough padding to match long rows.

For every resulting batch, both limits must hold:

```text
batch_size × longest source_length ≤ token budget
batch_size × longest complete_target_length ≤ token budget
```

The frozen campaign sets each budget to 4,096 padded positions.


In [4]:
#| export
def _batch_encoded_buffer(
    encoded_buffer: list[tuple[list[int], list[int]]],
    token_budget: int,
    pad_token_id: int,
) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
    """Length-group and batch one bounded encoded buffer."""
    # Each buffer item is two variable-length lists: (source_length,), (target_length,).
    encoded_buffer.sort(
        key=lambda pair: (
            max(len(pair[0]), len(pair[1])),
            len(pair[0]),
            len(pair[1]),
        )
    )

    batches = make_token_budget_batches(
        encoded_buffer,
        token_budget=token_budget,
    )

    for batch in batches:
        # Shapes: (batch, source_length), (batch, target_length + 1).
        yield _collate_token_batch(
            batch,
            pad_token_id,
        )

### Stream eligible examples

`iter_token_batches` is the public coordinator for the previous two helpers. Its purpose is to turn a large stream of WMT text into one padded tensor batch at a time without retaining the whole corpus in memory.

For one input example, it performs this pipeline:

```text
ParallelExample(source_text, target_text)
                 │
                 ├─ tokenize source + BOS/EOS → list[int]
                 ├─ tokenize target + BOS/EOS → list[int]
                 ├─ skip if either list exceeds 256 positions
                 └─ place the pair in the current encoded buffer
                                      │
                          buffer reaches 10,000 pairs
                                      │
                          sort and token-budget batch
                                      │
                 yield (source tensor, complete-target tensor)
```

A yielded result may look like this:

```text
source_token_ids: shape (batch=32, source_length=96)
target_token_ids: shape (batch=32, target_length + 1=104)

source storage = 32 × 96  = 3,072 positions ≤ 4,096
target storage = 32 × 104 = 3,328 positions ≤ 4,096
```

The next yielded batch may have a different `batch`, `source_length`, and `target_length + 1`; only the per-side budget is fixed. `maximum_examples` stops after the frozen number of eligible pairs, `maximum_length` rejects oversized sequences, `token_budget` controls padded batch storage, and `buffer_size` limits how many encoded examples are held for local length sorting.

The function does not move tensors to CUDA, calculate loss, or update parameters. Those responsibilities remain in the campaign training loop.


In [5]:
#| export
from collections.abc import Iterable


def iter_token_batches(
    examples: Iterable[ParallelExample],
    tokenizer: Tokenizer,
    *,
    maximum_examples: int,
    maximum_length: int,
    token_budget: int,
    buffer_size: int = 10_000,
) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
    """Encode eligible examples and yield padded token-budget batches."""
    if maximum_examples < 1:
        raise ValueError("maximum_examples must be positive")
    if maximum_length < 2:
        raise ValueError("maximum_length must be at least two")
    if buffer_size < 1:
        raise ValueError("buffer_size must be positive")

    pad_token_id = tokenizer.token_to_id("<pad>")
    bos_token_id = tokenizer.token_to_id("<bos>")
    eos_token_id = tokenizer.token_to_id("<eos>")

    if pad_token_id is None:
        raise ValueError("tokenizer is missing <pad>")
    if bos_token_id is None:
        raise ValueError("tokenizer is missing <bos>")
    if eos_token_id is None:
        raise ValueError("tokenizer is missing <eos>")

    # The buffer contains variable-length pairs, not rectangular tensors yet.
    encoded_buffer: list[tuple[list[int], list[int]]] = []
    eligible_examples = 0

    for example in examples:
        # source_ids has logical shape (source_length,).
        source_ids = [
            bos_token_id,
            *tokenizer.encode(example.source_text).ids,
            eos_token_id,
        ]
        # Complete target shape: (target_length + 1,) before decoder shifting.
        target_ids = [
            bos_token_id,
            *tokenizer.encode(example.target_text).ids,
            eos_token_id,
        ]

        if len(source_ids) > maximum_length or len(target_ids) > maximum_length:
            continue

        encoded_buffer.append((source_ids, target_ids))
        eligible_examples += 1

        if len(encoded_buffer) == buffer_size:
            yield from _batch_encoded_buffer(
                encoded_buffer,
                token_budget,
                pad_token_id,
            )
            encoded_buffer = []

        if eligible_examples == maximum_examples:
            break

    if encoded_buffer:
        yield from _batch_encoded_buffer(
            encoded_buffer,
            token_budget,
            pad_token_id,
        )
    if eligible_examples < maximum_examples:
        raise ValueError(
            f"requested {maximum_examples:,} eligible examples, but found {eligible_examples:,}"
        )

### Visible batching example

Three short translation pairs demonstrate that the iterator produces one source matrix and one complete-target matrix per batch. The small token budget forces more than one batch and makes the per-side budget visible.


In [6]:
canonical_tokenizer = Tokenizer.from_file(
    str(PROJECT_ROOT / "artifacts" / "tokenizers" / "wmt14-en-de-shared-bpe-37000.json")
)

fixture_examples = [
    ParallelExample(source_text="Hello.", target_text="Hallo."),
    ParallelExample(source_text="I agree.", target_text="Ich stimme zu."),
    ParallelExample(source_text="Thank you.", target_text="Vielen Dank."),
]

fixture_batches = list(
    iter_token_batches(
        fixture_examples,
        canonical_tokenizer,
        maximum_examples=3,
        maximum_length=16,
        token_budget=12,
        buffer_size=3,
    )
)

for batch_index, (source_token_ids, target_token_ids) in enumerate(
    fixture_batches,
    start=1,
):
    print(f"batch {batch_index}")
    print(f"  source shape: {tuple(source_token_ids.shape)}")
    print(source_token_ids)
    print(f"  complete-target shape: {tuple(target_token_ids.shape)}")
    print(target_token_ids)

batch 1
  source shape: (2, 4)
tensor([[    2, 25626,  5307,     3],
        [    2,  8821,  9405,     3]])
  complete-target shape: (2, 4)
tensor([[    2, 12909,  5307,     3],
        [    2, 12672, 36543,     3]])
batch 2
  source shape: (1, 5)
tensor([[   2, 3829, 5092, 4083,    3]])
  complete-target shape: (1, 5)
tensor([[    2,  4277, 12516, 12261,     3]])


### Focused batching assertions

The fixture must produce two batches containing all three examples. Both sides of every batch must stay within the 12-position demonstration budget.


In [7]:
assert len(fixture_batches) == 2
assert sum(source.shape[0] for source, _ in fixture_batches) == 3

for source_token_ids, target_token_ids in fixture_batches:
    assert source_token_ids.numel() <= 12
    assert target_token_ids.numel() <= 12
    assert source_token_ids.dtype == torch.long
    assert target_token_ids.dtype == torch.long

## Load verified WMT training data

Before constructing the full campaign iterator, a 16-example smoke run verifies
that the manifest, shared tokenizer, length limit, and frozen token budget work
together on real training sentences.

In [8]:
data_root = PROJECT_ROOT / "data" / "wmt14_en_de"
manifest_path = data_root / "manifests" / "wmt14-en-de-shard-100000.json"
manifest = load_manifest(manifest_path)

smoke_batches = list(
    iter_token_batches(
        iter_manifest_examples(
            data_root,
            manifest,
            split="train",
        ),
        canonical_tokenizer,
        maximum_examples=16,
        maximum_length=CANONICAL_CAMPAIGN.max_sequence_length,
        token_budget=CANONICAL_CAMPAIGN.token_budget_per_side,
        buffer_size=16,
    )
)

for batch_index, (source_token_ids, target_token_ids) in enumerate(
    smoke_batches,
    start=1,
):
    print(
        f"batch {batch_index}: "
        f"source={tuple(source_token_ids.shape)}, "
        f"complete_target={tuple(target_token_ids.shape)}"
    )

batch 1: source=(16, 55), complete_target=(16, 67)


In [9]:
assert sum(source.shape[0] for source, _ in smoke_batches) == 16

for source_token_ids, target_token_ids in smoke_batches:
    assert source_token_ids.numel() <= CANONICAL_CAMPAIGN.token_budget_per_side
    assert target_token_ids.numel() <= CANONICAL_CAMPAIGN.token_budget_per_side

## Required deliverables

- Preflight/resume runbook
- Session audit records
- Frozen-budget curves
- Validation-selected candidate


## Planned implementation sections

1. Paper and contract references
2. Typed implementation
3. Focused tests
4. Deterministic visible result
5. Exported API and artifact identities


## Explicitly deferred

Final-test tuning, Hub publication, and Gradio deployment.


## HITL checkpoint

Approve launch, continuations, anomalies, and validation-based selection.
